# T-05 — Phase 2: Evaluation, Error Analysis, and Demo

This separate notebook follows the Phase 2 requirements of *YadYar Lite*. It loads the exact artifacts produced in Phase 1, evaluates the unchanged baseline, creates understandable breakdowns and representative errors, runs a lightweight demo, and exports all raw outputs needed for the final report.

> Before running this notebook, run Phase 1 and keep `T05_Phase1_Outputs.zip`.


## What this notebook produces

- the two primary metrics: SQuAD Exact Match and Token F1;
- retrieval and abstention diagnostics;
- breakdowns by answerability, question length, and retrieval success;
- representative failure examples;
- a manual-review sheet for careful hallucination labeling;
- a three-question RAG demo;
- a simple future-integration JSON contract;
- one ZIP containing every result needed to build the final report and presentation.

The code never defines every wrong answer as hallucination. Automatic categories are only candidates; manual evidence review is required for a defensible unsupported-generation claim.


In [ ]:
%pip -q install \
    datasets==5.0.1 \
    sentence-transformers==5.7.0 \
    faiss-cpu==1.15.0 \
    transformers==5.15.0 \
    sentencepiece==0.2.2 \
    matplotlib==3.10.8


In [ ]:
from pathlib import Path
from collections import Counter
import io
import json
import math
import re
import shutil
import string
import zipfile

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

PHASE1_DIR = Path("/content/T05_Phase1_Outputs")
OUTPUT_DIR = Path("/content/T05_Phase2_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not (PHASE1_DIR / "phase1_config.json").exists():
    print("Upload T05_Phase1_Outputs.zip")
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError as exc:
        raise FileNotFoundError(
            "Place T05_Phase1_Outputs.zip in /content before running this cell."
        ) from exc

    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("No ZIP file was uploaded.")
    PHASE1_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        archive.extractall(PHASE1_DIR)

required = [
    "phase1_config.json", "phase1_questions.csv", "phase1_corpus.csv",
    "corpus_embeddings.npy", "corpus.faiss",
]
missing = [name for name in required if not (PHASE1_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing Phase 1 artifacts: {missing}")

print("Phase 1 artifacts loaded from:", PHASE1_DIR)


## 1. Restore the unchanged Phase 1 baseline

The question sample, corpus, saved vectors, FAISS index, model names, seed, and top-k setting all come from Phase 1. This prevents accidental changes between the two phases.


In [ ]:
with open(PHASE1_DIR / "phase1_config.json", encoding="utf-8") as file:
    config = json.load(file)

questions = pd.read_csv(PHASE1_DIR / "phase1_questions.csv")
corpus = pd.read_csv(PHASE1_DIR / "phase1_corpus.csv")
questions["gold_answers"] = questions["gold_answers_json"].apply(json.loads)
questions["is_answerable"] = questions["is_answerable"].astype(bool)

TOP_K = int(config["retrieval_top_k"])
EMBEDDING_MODEL = config["embedding_model"]
GENERATOR_MODEL = config["generator_model"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
index = faiss.read_index(str(PHASE1_DIR / "corpus.faiss"))

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
generator = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL).to(DEVICE)
generator.eval()

print("Device:", DEVICE)
print("Questions:", len(questions))
print("Corpus passages:", len(corpus))
print("Indexed passages:", index.ntotal)


## 2. Retrieve evidence for every question

We retrieve the top 3 passages for diagnostics, but the generator sees only rank 1. This keeps the baseline easy to explain and lets us distinguish a complete retrieval miss from a ranking error.


In [ ]:
query_embeddings = embedder.encode(
    questions["question"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype("float32")

similarities, positions = index.search(query_embeddings, TOP_K)

retrieval_rows = []
for row_number, question_row in questions.iterrows():
    selected = corpus.iloc[positions[row_number]].reset_index(drop=True)
    retrieved_ids = selected["context_id"].tolist()
    retrieval_rows.append({
        "id": question_row["id"],
        "retrieved_context_ids_json": json.dumps(retrieved_ids),
        "retrieved_titles_json": json.dumps(selected["title"].tolist()),
        "retrieval_scores_json": json.dumps(similarities[row_number].tolist()),
        "top1_context_id": retrieved_ids[0],
        "top1_title": selected.iloc[0]["title"],
        "top1_context": selected.iloc[0]["context"],
        "top1_similarity": float(similarities[row_number][0]),
        "retrieval_hit_top1": question_row["gold_context_id"] == retrieved_ids[0],
        "retrieval_hit_top3": question_row["gold_context_id"] in retrieved_ids,
    })

retrieval_df = pd.DataFrame(retrieval_rows)
results = questions.merge(retrieval_df, on="id", validate="one_to_one")

answerable_results = results[results["is_answerable"]]
print("Answerable retrieval hit@1:", round(answerable_results["retrieval_hit_top1"].mean(), 3))
print("Answerable retrieval hit@3:", round(answerable_results["retrieval_hit_top3"].mean(), 3))
display(results[["question", "top1_title", "top1_similarity", "retrieval_hit_top1"]].head())


## 3. Generate answers in batches

The decoding is deterministic (`do_sample=False`) and limited to 32 new tokens. The prompt asks for a short answer copied from evidence when possible and uses one clear abstention token.


In [ ]:
def build_prompt(question, passage):
    return f"""Answer the question using ONLY the passage below.
If the passage does not contain enough evidence, output exactly NO_ANSWER.
Keep the answer short and copy the answer span when possible.

Passage:
{passage}

Question: {question}
Answer:"""

def standardize_answer(text):
    cleaned = text.strip()
    no_answer_forms = {"no_answer", "no answer", "no-answer", "none"}
    return "NO_ANSWER" if cleaned.lower() in no_answer_forms else cleaned

def generate_in_batches(prompts, batch_size=8):
    predictions = []
    for start in range(0, len(prompts), batch_size):
        batch = prompts[start:start + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(DEVICE)
        with torch.inference_mode():
            output_ids = generator.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                num_beams=1,
            )
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        predictions.extend(standardize_answer(text) for text in decoded)
    return predictions

prompts = [
    build_prompt(row.question, row.top1_context)
    for row in results.itertuples(index=False)
]
results["prediction"] = generate_in_batches(prompts)
results["abstained"] = results["prediction"].eq("NO_ANSWER")
results["prediction_for_metric"] = results["prediction"].where(
    ~results["abstained"], ""
)

display(results[["question", "gold_answers", "prediction", "abstained"]].head(8))


## 4. Compute the two primary metrics

The functions below reproduce standard SQuAD normalization: lowercase, remove punctuation and English articles, and collapse whitespace. Unanswerable examples use an empty gold answer, so a correct `NO_ANSWER` abstention receives full credit.


In [ ]:
def normalize_answer(text):
    def remove_articles(value):
        return re.sub(r"\b(a|an|the)\b", " ", value)

    def remove_punctuation(value):
        return "".join(character for character in value if character not in string.punctuation)

    return " ".join(remove_articles(remove_punctuation(text.lower())).split())

def exact_match_score(prediction, gold_answers):
    golds = gold_answers if gold_answers else [""]
    return int(any(normalize_answer(prediction) == normalize_answer(gold) for gold in golds))

def token_f1_score(prediction, gold_answers):
    golds = gold_answers if gold_answers else [""]
    prediction_tokens = normalize_answer(prediction).split()
    scores = []

    for gold in golds:
        gold_tokens = normalize_answer(gold).split()
        common = Counter(prediction_tokens) & Counter(gold_tokens)
        shared = sum(common.values())

        if not prediction_tokens and not gold_tokens:
            scores.append(1.0)
        elif not prediction_tokens or not gold_tokens or shared == 0:
            scores.append(0.0)
        else:
            precision = shared / len(prediction_tokens)
            recall = shared / len(gold_tokens)
            scores.append(2 * precision * recall / (precision + recall))

    return max(scores)

results["exact_match"] = results.apply(
    lambda row: exact_match_score(row["prediction_for_metric"], row["gold_answers"]),
    axis=1,
)
results["token_f1"] = results.apply(
    lambda row: token_f1_score(row["prediction_for_metric"], row["gold_answers"]),
    axis=1,
)

primary_metrics = {
    "total_examples": int(len(results)),
    "exact_match_percent": round(100 * results["exact_match"].mean(), 2),
    "token_f1_percent": round(100 * results["token_f1"].mean(), 2),
}

retrieval_diagnostics = {
    "answerable_examples": int(results["is_answerable"].sum()),
    "retrieval_hit_at_1_percent": round(
        100 * results.loc[results["is_answerable"], "retrieval_hit_top1"].mean(), 2
    ),
    "retrieval_hit_at_3_percent": round(
        100 * results.loc[results["is_answerable"], "retrieval_hit_top3"].mean(), 2
    ),
    "unanswerable_examples": int((~results["is_answerable"]).sum()),
    "unanswerable_abstention_accuracy_percent": round(
        100 * results.loc[~results["is_answerable"], "abstained"].mean(), 2
    ),
}

print("Primary metrics")
display(pd.DataFrame([primary_metrics]))
print("Diagnostics")
display(pd.DataFrame([retrieval_diagnostics]))


## 5. Build simple, interpretable breakdowns

The breakdowns answer three practical questions:

1. Does performance change when the question is unanswerable?
2. Does question length matter?
3. When the gold passage is rank 1, does the generator still fail?


In [ ]:
median_question_length = int(results["question_word_count"].median())
results["question_length_group"] = np.where(
    results["question_word_count"] <= median_question_length,
    "short",
    "long",
)

def summarize_slice(frame, slice_name, group_name):
    return {
        "slice": slice_name,
        "group": group_name,
        "count": int(len(frame)),
        "exact_match_percent": round(100 * frame["exact_match"].mean(), 2),
        "token_f1_percent": round(100 * frame["token_f1"].mean(), 2),
        "abstention_rate_percent": round(100 * frame["abstained"].mean(), 2),
    }

breakdown_rows = []
for value, frame in results.groupby("is_answerable"):
    breakdown_rows.append(summarize_slice(
        frame, "answerability", "answerable" if value else "unanswerable"
    ))

for value, frame in results.groupby("question_length_group"):
    breakdown_rows.append(summarize_slice(frame, "question_length", value))

answerable_only = results[results["is_answerable"]]
for value, frame in answerable_only.groupby("retrieval_hit_top1"):
    breakdown_rows.append(summarize_slice(
        frame, "answerable_retrieval_top1", "hit" if value else "miss"
    ))

breakdowns = pd.DataFrame(breakdown_rows)
display(breakdowns)


## 6. Separate failure stages without overclaiming hallucination

Automatic categories help choose examples for inspection:

- **retrieval_miss:** gold passage not in top 3;
- **retrieval_ranking_error:** gold passage appears in top 3 but not rank 1;
- **ignored_evidence_candidate:** gold passage is rank 1 but the answer has zero token overlap;
- **partial_or_paraphrase:** some overlap but not exact;
- **absent_evidence_candidate:** an unanswerable question received a non-abstaining answer;
- **correct / correct_abstention:** the automatic metric accepted the output.

Only manual inspection can confirm whether a candidate output is truly unsupported by the retrieved evidence.


In [ ]:
def assign_failure_category(row):
    if not row["is_answerable"]:
        return "correct_abstention" if row["abstained"] else "absent_evidence_candidate"
    if row["exact_match"]:
        return "correct"
    if not row["retrieval_hit_top3"]:
        return "retrieval_miss"
    if not row["retrieval_hit_top1"]:
        return "retrieval_ranking_error"
    if row["token_f1"] > 0:
        return "partial_or_paraphrase"
    return "ignored_evidence_candidate"

results["automatic_category"] = results.apply(assign_failure_category, axis=1)

category_counts = (
    results["automatic_category"]
    .value_counts()
    .rename_axis("automatic_category")
    .reset_index(name="count")
)
display(category_counts)

failure_rows = results[~results["automatic_category"].isin(["correct", "correct_abstention"])]
representative_errors = (
    failure_rows.groupby("automatic_category", group_keys=False)
    .head(3)
    [[
        "id", "automatic_category", "question", "gold_answers_json",
        "prediction", "top1_title", "top1_context", "top1_similarity",
        "retrieval_hit_top1", "retrieval_hit_top3", "token_f1",
    ]]
    .reset_index(drop=True)
)
display(representative_errors[[
    "automatic_category", "question", "gold_answers_json", "prediction"
]])


## 7. Prepare a small manual-review sheet

The exported sheet contains at most 24 balanced candidate failures. Later, review each row against `top1_context` and fill:

- `manual_label`: `correct`, `partially_correct`, `unsupported_generation`, `retrieval_error`, `unanswerable_failure`, `multi_answer_collapse`, or `other`;
- `answer_supported_by_top1`: `1` or `0`;
- `review_note`: one short explanation.

Do not fill this now if you plan to send the raw ZIP back for report creation; the reviewer can complete it from the saved evidence.


In [ ]:
review_parts = []
for _, frame in failure_rows.groupby("automatic_category"):
    review_parts.append(frame.head(6))

review_columns = [
    "id", "automatic_category", "question", "gold_answers_json",
    "prediction", "top1_title", "top1_context", "top1_similarity",
    "retrieval_hit_top1", "retrieval_hit_top3", "exact_match", "token_f1",
]
if review_parts:
    manual_review = pd.concat(review_parts, ignore_index=True).head(24)[review_columns]
else:
    manual_review = results.head(0)[review_columns].copy()
manual_review["manual_label"] = ""
manual_review["answer_supported_by_top1"] = ""
manual_review["review_note"] = ""

print("Rows selected for manual review:", len(manual_review))
display(manual_review[[
    "automatic_category", "question", "prediction", "manual_label"
]].head(10))


## 8. Lightweight demo

This is intentionally a notebook function, not a deployed product. It accepts a question, retrieves one passage, and returns a compact dictionary containing the answer, abstention decision, source, and similarity score.


In [ ]:
def answer_one_question(question):
    query = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, found_positions = index.search(query, 1)
    passage = corpus.iloc[int(found_positions[0][0])]
    prediction = generate_in_batches([
        build_prompt(question, passage["context"])
    ], batch_size=1)[0]
    return {
        "question": question,
        "answer": prediction,
        "abstained": prediction == "NO_ANSWER",
        "source_title": passage["title"],
        "source_context_id": passage["context_id"],
        "retrieval_similarity": round(float(scores[0][0]), 4),
    }

demo_questions = [
    results[results["is_answerable"]].iloc[0]["question"],
    results[~results["is_answerable"]].iloc[0]["question"],
    "What is the instructor's private email address?",
]
demo_outputs = pd.DataFrame([answer_one_question(q) for q in demo_questions])
display(demo_outputs)


## 9. Limitations, future work, and integration contract

### Honest limitations

- The corpus and question sample are small and come from one English benchmark.
- SQuAD EM/F1 measure answer overlap, not factual support by themselves.
- A gold-context hit does not prove that the generated answer is supported.
- FLAN-T5-base receives only one retrieved passage and has no support gate.
- Results do not establish a causal comparison with a no-RAG system.

### Realistic future work

1. Add a small reranker or a stronger embedding model for ranking errors.
2. Add an entailment/claim-support gate that forces abstention when evidence is insufficient.

The following JSON contract shows how this component could later connect to a lightweight learning assistant. No server or deployment is required.


In [ ]:
integration_contract = {
    "component": "T05_lightweight_rag",
    "input": {
        "question": "string",
        "top_k": 3,
    },
    "output": {
        "answer": "string or NO_ANSWER",
        "abstained": "boolean",
        "source_title": "string",
        "source_context_id": "string",
        "retrieval_similarity": "float",
    },
    "future_partners": [
        "T-06 document parser as a corpus source",
        "T-04 dialogue component as a question source",
        "T-26 unsupported-claim detector as an abstention gate",
    ],
}
display(integration_contract)


## 10. Save every raw output for the report

The final ZIP contains metrics, all predictions and retrieved evidence, breakdowns, error examples, the manual-review sheet, demo outputs, the integration contract, and a summary figure. Send this ZIP together with the Phase 1 ZIP when requesting the reports and presentation.


In [ ]:
results_to_save = results.copy()
results_to_save["gold_answers"] = results_to_save["gold_answers"].apply(json.dumps)
results_to_save.to_csv(OUTPUT_DIR / "phase2_predictions.csv", index=False)
breakdowns.to_csv(OUTPUT_DIR / "phase2_breakdowns.csv", index=False)
category_counts.to_csv(OUTPUT_DIR / "phase2_category_counts.csv", index=False)
representative_errors.to_csv(OUTPUT_DIR / "phase2_representative_errors.csv", index=False)
manual_review.to_csv(OUTPUT_DIR / "phase2_manual_review_template.csv", index=False)
demo_outputs.to_csv(OUTPUT_DIR / "phase2_demo_outputs.csv", index=False)

metrics = {
    "primary_metrics": primary_metrics,
    "retrieval_and_abstention_diagnostics": retrieval_diagnostics,
    "question_length_median_words": median_question_length,
    "automatic_category_counts": dict(zip(
        category_counts["automatic_category"],
        category_counts["count"].astype(int),
    )),
    "important_interpretation": (
        "Automatic error categories are candidates. Wrong answers are not "
        "automatically labeled as hallucinations; evidence support requires manual review."
    ),
}
with open(OUTPUT_DIR / "phase2_metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / "integration_contract.json", "w", encoding="utf-8") as file:
    json.dump(integration_contract, file, indent=2, ensure_ascii=False)

notes = {
    "limitations": [
        "Small English SQuAD v2 sample and a controlled corpus.",
        "EM/F1 do not directly prove evidential support.",
        "Only the top-ranked passage is given to a small generator.",
        "No causal no-RAG comparison is claimed.",
    ],
    "future_work": [
        "Try a lightweight reranker or stronger embedding model.",
        "Add an entailment-based support gate before returning an answer.",
    ],
    "ai_assistance_acknowledgement_required": True,
}
with open(OUTPUT_DIR / "phase2_notes.json", "w", encoding="utf-8") as file:
    json.dump(notes, file, indent=2, ensure_ascii=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

answerability_plot = breakdowns[breakdowns["slice"] == "answerability"].set_index("group")
answerability_plot[["exact_match_percent", "token_f1_percent"]].plot(
    kind="bar", ax=axes[0], color=["#31688e", "#35b779"]
)
axes[0].set_title("Answer quality by answerability")
axes[0].set_xlabel("")
axes[0].set_ylabel("Percent")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(["Exact Match", "Token F1"], frameon=False)

failure_counts = category_counts[
    ~category_counts["automatic_category"].isin(["correct", "correct_abstention"])
]
axes[1].barh(
    failure_counts["automatic_category"],
    failure_counts["count"],
    color="#f28e2b",
)
axes[1].set_title("Automatic candidate failure counts")
axes[1].set_xlabel("Examples")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "phase2_summary_figure.png", dpi=180, bbox_inches="tight")
plt.show()

manifest = {
    "files": sorted(path.name for path in OUTPUT_DIR.iterdir()),
    "phase1_seed": config["seed"],
    "models": {
        "embedding": EMBEDDING_MODEL,
        "generator": GENERATOR_MODEL,
    },
}
with open(OUTPUT_DIR / "phase2_manifest.json", "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

archive = shutil.make_archive(
    "/content/T05_Phase2_Outputs", "zip", root_dir=OUTPUT_DIR
)
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download manually from:", archive)


## Phase 2 is complete

Please keep and send both files without renaming their internal contents:

1. `T05_Phase1_Outputs.zip`
2. `T05_Phase2_Outputs.zip`

The final Phase 1 report (maximum 6 pages), Phase 2 report (maximum 10 pages), and short presentation should be written only after these real outputs are available.
